In [ ]:
!apt-get install -y fonts-nanum > /dev/null 2>&1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import joblib
import os

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, roc_auc_score,
                             precision_recall_curve, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import lightgbm as lgb


fm._load_fontmanager(try_read_cache=False)

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# models 폴더 자동 생성
os.makedirs('../models', exist_ok=True)

In [ ]:
df = pd.read_excel('pitcher_stats_all.xlsx')
print(f"전체 shape: {df.shape}")
print(f"\n[지명여부 분포]")
print(df['지명여부'].value_counts())
print(f"\n[연도별 지명 현황]")
print(df.groupby('year')['지명여부']
      .apply(lambda x: (x == 'O').sum())
      .to_frame('지명자 수'))

In [ ]:
# 1) KBO에서 BB가 0이라 계산 불가능한 KBB → 0으로 대체
df['KBB'] = df['KBB'].fillna(0)

# 2) 비율 지표 → 평균값 대체
ratio_cols = ['BAA', 'OBP_against', 'SLG_against', 'OPS_against',
              'BABIP_against', 'LOB_pct',
              'GB_pct', 'FB_pct', 'LD_pct']
for col in ratio_cols:
    df[col] = df[col].fillna(df[col].mean())

# 3) 카운트형 결측 → 0으로 대체
count_cols = ['GB', 'FB', 'LD', 'BIP']
for col in count_cols:
    df[col] = df[col].fillna(0)

# 4) player_id 결측 → 임의 ID 부여 (StratifiedGroupKFold용)
n_missing = df['player_id'].isnull().sum()
df.loc[df['player_id'].isnull(), 'player_id'] = range(
    900_000_000_000,
    900_000_000_000 + n_missing
)

print(f"결측치 처리 후 남은 결측 (지명 관련 컬럼 제외):")
remaining = df.drop(columns=['라운드','순위','포지션','지명구단']).isnull().sum()
print(remaining[remaining > 0])

In [ ]:
print(f"필터링 전: {len(df)}명")
print(f"  지명자: {(df['지명여부']=='O').sum()}명")

df = df[df['IP_decimal'] >= 5].copy().reset_index(drop=True)

print(f"\n필터링 후 (5이닝 이상): {len(df)}명")
print(f"  지명자: {(df['지명여부']=='O').sum()}명")
print(f"  지명률: {(df['지명여부']=='O').sum()/len(df)*100:.2f}%")

In [ ]:
features = [
    # 기본 카운트
    'G', 'GS', 'IP_decimal', 'TBF', 'W',
    # 결과
    'H', 'HR', 'BB', 'SO',
    # 핵심 비율
    'ERA', 'WHIP', 'BAA', 'OPS_against', 'K_pct',
    # 세이버매트릭스
    'kFIP', 'kxFIP', 'KBB', 'BB_pct',
    # 9이닝 환산
    'K9', 'BB9', 'HR9', 'H9',
    # 타구 분포
    'GB_pct', 'FB_pct', 'LD_pct',
]

df['지명_binary'] = (df['지명여부'] == 'O').astype(int)

X = df[features]
y = df['지명_binary']
groups = df['player_id']

print(f"피처 수: {len(features)}개")
print(f"X shape: {X.shape}")
print(f"y 분포: {dict(y.value_counts())}")

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        min_child_samples=10,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    ))
])

# OOF 예측 (player_id 그룹 기준)
oof_proba = cross_val_predict(
    pipeline, X, y,
    cv=sgkf.split(X, y, groups=groups),
    method='predict_proba',
    n_jobs=-1
)[:, 1]

print(f"AUROC (CV): {roc_auc_score(y, oof_proba):.4f}")

In [ ]:
precision_arr, recall_arr, thresholds = precision_recall_curve(y, oof_proba)

f1_arr = np.where(
    (precision_arr[:-1] + recall_arr[:-1]) == 0, 0,
    2 * precision_arr[:-1] * recall_arr[:-1]
      / (precision_arr[:-1] + recall_arr[:-1])
)

best_idx = f1_arr.argmax()
best_threshold = thresholds[best_idx]
best_f1 = f1_arr[best_idx]

print(f"최적 Threshold: {best_threshold:.3f}")
print(f"최적 F1-score: {best_f1:.4f}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(recall_arr, precision_arr, color='#378ADD', lw=2)
axes[0].scatter(recall_arr[best_idx], precision_arr[best_idx],
                color='#D85A30', s=120, zorder=5,
                label=f'최적 threshold = {best_threshold:.2f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(thresholds, f1_arr, color='#1D9E75', lw=2)
axes[1].axvline(best_threshold, color='#D85A30', ls='--')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1-score')
axes[1].set_title('Threshold별 F1-score', fontweight='bold')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
y_pred = (oof_proba >= best_threshold).astype(int)

print("=" * 50)
print("[ 투수 드래프트 예측 모델 — 최종 성능 ]")
print("=" * 50)

auroc  = roc_auc_score(y, oof_proba)
f1_mac = f1_score(y, y_pred, average='macro')
f1_pos = f1_score(y, y_pred, pos_label=1)

print(f"AUROC          : {auroc:.4f}")
print(f"F1 (macro)     : {f1_mac:.4f}")
print(f"F1 (지명 클래스): {f1_pos:.4f}")
print(f"Threshold      : {best_threshold:.3f}")

# Precision@K
def precision_at_k(y_true, y_proba, k):
    top_k = np.argsort(y_proba)[::-1][:k]
    return np.array(y_true)[top_k].sum()

print(f"\nPrecision@10 : {precision_at_k(y, oof_proba, 10)}명 / 10명")
print(f"Precision@20 : {precision_at_k(y, oof_proba, 20)}명 / 20명")
print(f"Precision@50 : {precision_at_k(y, oof_proba, 50)}명 / 50명")

# 분류 리포트
print("\n" + "=" * 50)
print(classification_report(y, y_pred, target_names=['미지명', '지명']))

# 혼동 행렬
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['미지명', '지명'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix (threshold={best_threshold:.2f})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
pipeline.fit(X, y)
model = pipeline.named_steps['model']

label_kor = {
    'G':'경기수','GS':'선발등판','IP_decimal':'이닝','TBF':'상대타자수','W':'승',
    'H':'피안타','HR':'피홈런','BB':'볼넷','SO':'탈삼진',
    'ERA':'평균자책점','WHIP':'WHIP','BAA':'피안타율','OPS_against':'피OPS',
    'K_pct':'K%','kFIP':'kFIP','kxFIP':'kxFIP','KBB':'K/BB','BB_pct':'BB%',
    'K9':'K/9','BB9':'BB/9','HR9':'HR/9','H9':'H/9',
    'GB_pct':'땅볼%','FB_pct':'뜬공%','LD_pct':'라인드라이브%',
}

imp_df = (pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
})
.sort_values('importance', ascending=True)
.tail(15))

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh([label_kor[f] for f in imp_df['feature']],
        imp_df['importance'], color='#1D9E75', edgecolor='none', height=0.65)
ax.set_title('투수 모델 — LightGBM 피처 중요도 Top 15', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(pipeline,       'pitcher_model.pkl')
joblib.dump(best_threshold, 'pitcher_threshold.pkl')
joblib.dump(features,       'pitcher_features.pkl')

print("✅ 투수 모델 저장 완료")
print(f"  - pitcher_model.pkl")
print(f"  - pitcher_threshold.pkl")
print(f"  - pitcher_features.pkl")